In [1]:
import pandas as pd
import requests
import json

In [15]:
url = 'https://raw.githubusercontent.com/valentinesvev/hackaton_project/main/data/raw/base.csv' # таблица из условия

In [16]:
base = pd.read_csv(url, on_bad_lines='warn')

In [17]:
base

,Номер студента,Сумма,Курс,Время
0,437,"8 950,00",ML про,04.08.2026 09:08:31
1,437,"7 890,00",Аналитика старт,04.08.2026 09:35:00
2,443,"8 475,00",Линейная алгебра,04.08.2026 10:14:38
3,443,"8 475,00",Мат анализ,04.08.2026 10:14:38
4,430,"8 950,00",К ВУЗу,04.08.2026 13:38:06
...,...,...,...,...
790,36,"8 950,00",Линейная алгебра,09.09.2026 19:26:20
791,36,"8 950,00",Мат анализ,09.09.2026 19:26:20
792,36,"8 950,00",Теория вероятностей,09.09.2026 19:26:20
793,407,"8 950,00",Линейная алгебра,09.09.2026 23:11:20


In [8]:
url = "https://raw.githubusercontent.com/valentinesvev/hackaton_project/main/data/raw/tg_posts_text.json" # выгрузка постов поступашек

data = requests.get(url).json()

print(type(data))

if isinstance(data, dict):
    print(data.keys())

<class 'dict'>
dict_keys(['name', 'type', 'id', 'messages'])


In [9]:
posts = data["messages"]

In [20]:
data['messages'][25]

{'id': 27,
 'type': 'message',
 'date': '2022-02-19T09:54:18',
 'date_unixtime': '1645253658',
 'edited': '2022-04-18T07:02:34',
 'edited_unixtime': '1650254554',
 'from': 'Поступашки - ШАД, Стажировки и Магистратура',
 'from_id': 'channel1756387595',
 'photo': '(File not included. Change data exporting settings to download.)',
 'photo_file_size': 60318,
 'width': 900,
 'height': 500,
 'text': 'Физтехи Lives Matter ✊',
 'text_entities': [{'type': 'plain', 'text': 'Физтехи Lives Matter ✊'}],
 'reactions': [{'type': 'emoji', 'count': 6, 'emoji': '👍'}]}

In [21]:
df = pd.json_normalize(posts)

print(df.columns.tolist())
df.head()

['id', 'type', 'date', 'date_unixtime', 'actor', 'actor_id', 'action', 'title', 'text', 'text_entities', 'photo', 'photo_file_size', 'width', 'height', 'edited', 'edited_unixtime', 'from', 'from_id', 'reactions', 'file', 'file_name', 'file_size', 'media_type', 'mime_type', 'duration_seconds', 'thumbnail', 'thumbnail_file_size', 'message_id', 'forwarded_from', 'forwarded_from_id', 'poll.question', 'poll.closed', 'poll.total_voters', 'poll.answers', 'inline_bot_buttons']


,id,type,date,date_unixtime,actor,actor_id,action,title,text,text_entities,...,thumbnail,thumbnail_file_size,message_id,forwarded_from,forwarded_from_id,poll.question,poll.closed,poll.total_voters,poll.answers,inline_bot_buttons
0,1,service,2022-01-31T14:12:32,1643627552,"Поступашки - ШАД, Стажировки и Магистратура",channel1756387595,create_channel,"Поступашки - ШАД, Стажировки",,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,service,2022-01-31T14:12:32,1643627552,"Поступашки - ШАД, Стажировки и Магистратура",channel1756387595,edit_group_photo,NaN,,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,service,2022-02-01T13:40:33,1643712033,"Поступашки - ШАД, Стажировки и Магистратура",channel1756387595,edit_group_title,"Поступашки - ШАД, Стажировки и Магистратура",,[],...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,5,message,2022-02-12T20:26:12,1644686772,NaN,NaN,NaN,NaN,"[Наслаждаемся моим первым опытом 😎, {'type': ...","[{'type': 'plain', 'text': 'Наслаждаемся моим ...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6,message,2022-02-13T21:11:10,1644775870,NaN,NaN,NaN,NaN,Школа Анализа Данных не одна?😱 \n \nВсё слышал...,"[{'type': 'plain', 'text': 'Школа Анализа Данн...",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
def clean_text(value):
    if isinstance(value, list):
        return " ".join(
            item.get("text", "") if isinstance(item, dict) else str(item)
            for item in value
        )
    return str(value) if value is not None else ""

events = pd.DataFrame({
    "event_id": "tg_post_" + df["id"].astype(str),
    "event_type": "channel_post_published",
    "event_timestamp": pd.to_datetime(df["date"]),
    "channel": "postupashki",
    "post_id": df["id"],
    "post_text": df["text"].apply(clean_text),
    "views": df["views"] if "views" in df.columns else None,
    "forwards": df["forwards"] if "forwards" in df.columns else None,
    "source": "telegram_export",
    "is_real_data": True
})

events.head()

,event_id,event_type,event_timestamp,channel,post_id,post_text,views,forwards,source,is_real_data
0,tg_post_1,channel_post_published,2022-01-31 14:12:32,postupashki,1,,None,None,telegram_export,True
1,tg_post_2,channel_post_published,2022-01-31 14:12:32,postupashki,2,,None,None,telegram_export,True
2,tg_post_3,channel_post_published,2022-02-01 13:40:33,postupashki,3,,None,None,telegram_export,True
3,tg_post_5,channel_post_published,2022-02-12 20:26:12,postupashki,5,Наслаждаемся моим первым опытом 😎 https://www...,None,None,telegram_export,True
4,tg_post_6,channel_post_published,2022-02-13 21:11:10,postupashki,6,Школа Анализа Данных не одна?😱 \n \nВсё слышал...,None,None,telegram_export,True


In [23]:
events.to_csv("own_channel_events.csv", index=False)

In [24]:
events["post_type"] = "content"

In [25]:
events

,event_id,event_type,event_timestamp,channel,post_id,post_text,views,forwards,source,is_real_data,post_type
0,tg_post_1,channel_post_published,2022-01-31 14:12:32,postupashki,1,,None,None,telegram_export,True,content
1,tg_post_2,channel_post_published,2022-01-31 14:12:32,postupashki,2,,None,None,telegram_export,True,content
2,tg_post_3,channel_post_published,2022-02-01 13:40:33,postupashki,3,,None,None,telegram_export,True,content
3,tg_post_5,channel_post_published,2022-02-12 20:26:12,postupashki,5,Наслаждаемся моим первым опытом 😎 https://www...,None,None,telegram_export,True,content
4,tg_post_6,channel_post_published,2022-02-13 21:11:10,postupashki,6,Школа Анализа Данных не одна?😱 \n \nВсё слышал...,None,None,telegram_export,True,content
...,...,...,...,...,...,...,...,...,...,...,...
1224,tg_post_1895,channel_post_published,2026-09-08 18:27:01,postupashki,1895,Собрали все задачи с алгосекции в Яндексе в од...,None,None,telegram_export,True,content
1225,tg_post_1897,channel_post_published,2026-09-09 17:17:20,postupashki,1897,Как готовиться к софт собесам в Авито \n\nОчен...,None,None,telegram_export,True,content
1226,tg_post_1898,channel_post_published,2026-09-10 11:59:39,postupashki,1898,"Как хакнуть HR \n\nВ рамках открытой недели ""...",None,None,telegram_export,True,content
1227,tg_post_1899,channel_post_published,2026-09-10 17:06:13,postupashki,1899,Продолжаем знакомиться с талантливыми ученикам...,None,None,telegram_export,True,content


In [26]:
# оставим только данные после 4 августа, тк ранее данных о продажах нет
analysis_posts = events[
    (events["event_timestamp"] >= "2026-08-04") &
    (events["event_timestamp"] <= "2026-09-10 23:59:59")
].copy()

analysis_posts.shape

(63, 11)

In [27]:
analysis_posts # нормальная таблица всех постов с 4 августа

,event_id,event_type,event_timestamp,channel,post_id,post_text,views,forwards,source,is_real_data,post_type
1165,tg_post_1829,channel_post_published,2026-08-04 18:50:24,postupashki,1829,"C какими айтишницами стоит строить отношения, ...",None,None,telegram_export,True,content
1166,tg_post_1830,channel_post_published,2026-08-05 19:03:53,postupashki,1830,Как повысить профессиональный рейтинг \n\nРан...,None,None,telegram_export,True,content
1167,tg_post_1831,channel_post_published,2026-08-06 19:23:59,postupashki,1831,Какие пет проекты гарантирует тебе оффер на мл...,None,None,telegram_export,True,content
1168,tg_post_1833,channel_post_published,2026-08-07 17:22:16,postupashki,1833,Слив алгоритмического собеса в Авито. Смотрим!...,None,None,telegram_export,True,content
1169,tg_post_1834,channel_post_published,2026-08-08 09:05:37,postupashki,1834,Магистратура по e-commerce от РУДН и Wildberri...,None,None,telegram_export,True,content
...,...,...,...,...,...,...,...,...,...,...,...
1223,tg_post_1894,channel_post_published,2026-09-07 18:51:57,postupashki,1894,,None,None,telegram_export,True,content
1224,tg_post_1895,channel_post_published,2026-09-08 18:27:01,postupashki,1895,Собрали все задачи с алгосекции в Яндексе в од...,None,None,telegram_export,True,content
1225,tg_post_1897,channel_post_published,2026-09-09 17:17:20,postupashki,1897,Как готовиться к софт собесам в Авито \n\nОчен...,None,None,telegram_export,True,content
1226,tg_post_1898,channel_post_published,2026-09-10 11:59:39,postupashki,1898,"Как хакнуть HR \n\nВ рамках открытой недели ""...",None,None,telegram_export,True,content


In [34]:
# поиск рекламных постов по ключевым словам

keywords = r"курс|набор|запуск|скидк|промокод|цена|стоимость|купить|записат|оплат|регистрац"

ads = analysis_posts[
    analysis_posts["post_text"].str.contains(keywords, case=False, na=False)
].copy()

ads.shape

(31, 11)

In [35]:
ads[["event_timestamp", "post_text"]]

,event_timestamp,post_text
1168,2026-08-07 17:22:16,Слив алгоритмического собеса в Авито. Смотрим!...
1169,2026-08-08 09:05:37,Магистратура по e-commerce от РУДН и Wildberri...
1170,2026-08-09 13:32:11,"Дайджест вакансий \n\nТоварищи, собрали свежие..."
1171,2026-08-09 18:35:40,"Как найти стажировку зарубежом \n\nТоварищи, п..."
1172,2026-08-10 16:26:59,Из процесс-менеджера в продуктового аналитика ...
1174,2026-08-10 20:33:06,"У России три пути: 18+, ***** и IT\n \nИ кажет..."
1176,2026-08-11 15:03:39,План на следующий учебный год: запустить IT-пр...
1177,2026-08-11 18:50:13,Как искать работу в 2026 году \n\nУже писал пр...
1178,2026-08-12 19:05:42,Гайд на собесы в СберЗвук Data Analyst \n\nВып...
1179,2026-08-13 10:05:42,"Товарищи, появился неплохой шанс прокачать сво..."
